# StormEngine V7-B workflow

390-point baseline: 239 physical DPC stations plus 151 model-derived Open-Meteo marine support points. Run one stage at a time.

In [4]:
from pathlib import Path
import subprocess, sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
CONFIG='configs/v7_b.yaml'
DEVICE='cuda'
RUN_PILOT=False 
RUN_FORMAL_TRAINING=False
RUN_2017_EVALUATION=False
RUN_COMBINED_REPLAY=True
SAVE_PREDICTIONS=False
def run(*args):
    command=[sys.executable,*map(str,args)]
    print(' '.join(command),flush=True)
    process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
    assert process.stdout is not None
    for line in process.stdout: print(line,end='',flush=True)
    code=process.wait()
    if code: raise subprocess.CalledProcessError(code,command)
print(ROOT)

D:\Documents\py_projects\StormEngine-DL\StormEngine-DL


## 1. Unit tests, ERA5 preflight, and real 390-point input check

In [2]:
run('-m','pytest','tests/test_v7_dataset.py','tests/test_v7_input.py','tests/test_v7_model.py','tests/test_open_meteo.py','-q')
run('scripts/check_v7.py','preflight','--config',CONFIG,'--device',DEVICE)
run('scripts/check_v7_b_input.py')

D:\anaconda3\envs\stormengine\python.exe -m pytest tests/test_v7_dataset.py tests/test_v7_input.py tests/test_v7_model.py tests/test_open_meteo.py -q
............                                                             [100%]
12 passed in 12.82s
D:\anaconda3\envs\stormengine\python.exe scripts/check_v7.py preflight --config configs/v7_b.yaml --device cuda
{
  "mode": "preflight",
  "device": "cuda",
  "splits": {
    "train_samples": 52567,
    "validation_samples": 8767,
    "test_years": [
      2017
    ]
  },
  "point_values": [
    16,
    12,
    390,
    5
  ],
  "value_mask": [
    16,
    12,
    390,
    5
  ],
  "observation_age": [
    16,
    12,
    390,
    5
  ],
  "prediction": [
    16,
    6,
    5,
    31,
    33
  ],
  "finite": true,
  "valid_fraction": 0.692481279373169,
  "stations_present_per_hour": {
    "min": 250,
    "median": 345.0,
    "max": 384
  },
  "nonzero_age_fraction_of_valid": 0.008122962899506092,
  "contract": {
    "version": "stormengine-

## 2. Smoke test and 200-batch benchmark

In [3]:
run('scripts/check_v7.py','smoke','--config',CONFIG,'--device',DEVICE)
run('scripts/check_v7.py','benchmark','--config',CONFIG,'--device',DEVICE,'--batches','200')

D:\anaconda3\envs\stormengine\python.exe scripts/check_v7.py smoke --config configs/v7_b.yaml --device cuda
Epoch 1/1 | train: 10 batches
  train 10/10 (100.0%) loss=0.90239 elapsed=0.0m ETA=0.0m
Epoch 1/1 | validation: 5 batches
  validation 5/5 (100.0%) elapsed=0.0m ETA=0.0m
{
  "mode": "smoke",
  "device": "cuda",
  "gpu": "NVIDIA GeForce RTX 4060 Laptop GPU",
  "contract": {
    "version": "stormengine-v7-mask-aware-v1",
    "input_variables": [
      "u10",
      "v10",
      "i10fg",
      "t2m",
      "tp"
    ],
    "include_age": true,
    "observation_age_units": "hours_capped_at_1",
    "station_count": 390,
    "cache_identity": "v7_cache_identity_2010_2017.json",
    "static_fields": "adriatic_390_fields.npz",
    "model": {
      "include_age": true,
      "point_hidden": 64,
      "latent_channels": 64,
      "processor_layers": 2,
      "kernel_size": 3,
      "gaussian_sigma": 0.1,
      "static_channels": 2,
      "point_static_channels": 2
    }
  },
  "validation_mi

## 3. Short pilot
Enable only after smoke and benchmark pass.

In [5]:
if RUN_PILOT:
    run('scripts/check_v7.py','pilot','--config',CONFIG,'--device',DEVICE,'--epochs','5')
else: print('Pilot skipped. Set RUN_PILOT=True when ready.')

D:\anaconda3\envs\stormengine\python.exe scripts/check_v7.py pilot --config configs/v7_b.yaml --device cuda --epochs 5
Epoch 1/5 | train: 300 batches
  train 100/300 ( 33.3%) loss=0.72316 elapsed=0.3m ETA=0.5m
  train 200/300 ( 66.7%) loss=0.64225 elapsed=0.5m ETA=0.3m
  train 300/300 (100.0%) loss=0.88646 elapsed=0.8m ETA=0.0m
Epoch 1/5 | validation: 75 batches
  validation 75/75 (100.0%) elapsed=0.1m ETA=0.0m
Epoch 1/5 complete | train_loss=0.826495 validation_loss=1.010849 best=1.010849 improved=True elapsed=0.9m max_remaining≈0.1h
Epoch 2/5 | train: 300 batches
  train 100/300 ( 33.3%) loss=0.68202 elapsed=0.3m ETA=0.5m
  train 200/300 ( 66.7%) loss=1.06049 elapsed=0.5m ETA=0.3m
  train 300/300 (100.0%) loss=0.46190 elapsed=0.9m ETA=0.0m
Epoch 2/5 | validation: 75 batches
  validation 75/75 (100.0%) elapsed=0.1m ETA=0.0m
Epoch 2/5 complete | train_loss=0.645548 validation_loss=0.889515 best=0.889515 improved=True elapsed=1.0m max_remaining≈0.0h
Epoch 3/5 | train: 300 batches
  trai

## 4. Formal 2010-2015 training
Enable only after reviewing pilot. Uses 2016 for model selection and saves best.pt/last.pt.

In [9]:
if RUN_FORMAL_TRAINING:
    run('scripts/check_v7.py','train','--config',CONFIG,'--device',DEVICE)
else: print('Formal training skipped.')

D:\anaconda3\envs\stormengine\python.exe scripts/check_v7.py train --config configs/v7_b.yaml --device cuda
Epoch 1/80 | train: 3286 batches
  train 100/3286 (  3.0%) loss=0.72316 elapsed=0.3m ETA=9.0m
  train 200/3286 (  6.1%) loss=0.64225 elapsed=0.6m ETA=8.6m
  train 300/3286 (  9.1%) loss=0.88646 elapsed=0.8m ETA=8.2m
  train 400/3286 ( 12.2%) loss=0.75153 elapsed=1.1m ETA=7.9m
  train 500/3286 ( 15.2%) loss=0.79644 elapsed=1.4m ETA=7.6m
  train 600/3286 ( 18.3%) loss=0.48249 elapsed=1.6m ETA=7.3m
  train 700/3286 ( 21.3%) loss=0.65502 elapsed=1.9m ETA=7.0m
  train 800/3286 ( 24.3%) loss=0.53782 elapsed=2.2m ETA=6.8m
  train 900/3286 ( 27.4%) loss=0.48671 elapsed=2.4m ETA=6.5m
  train 1000/3286 ( 30.4%) loss=0.52299 elapsed=2.7m ETA=6.2m
  train 1100/3286 ( 33.5%) loss=0.54814 elapsed=3.0m ETA=5.9m
  train 1200/3286 ( 36.5%) loss=0.47066 elapsed=3.3m ETA=5.7m
  train 1300/3286 ( 39.6%) loss=0.54783 elapsed=3.5m ETA=5.4m
  train 1400/3286 ( 42.6%) loss=0.46287 elapsed=3.8m ETA=5.1m


## 5. Frozen 2017 evaluation

In [3]:
if RUN_2017_EVALUATION:
    run('scripts/evaluate_v7_a.py','--config',CONFIG,'--checkpoint','artifacts/v7_b_2010_2017/best.pt','--scenario','clean','--device',DEVICE,'--output-dir','artifacts/v7_b_2010_2017/evaluation_2017_clean_seed42')
    for seed in (42,123,2026): run('scripts/evaluate_v7_a.py','--config',CONFIG,'--checkpoint','artifacts/v7_b_2010_2017/best.pt','--scenario','missing','--seed',str(seed),'--device',DEVICE,'--output-dir',f'artifacts/v7_b_2010_2017/evaluation_2017_missing_seed{seed}')
else: print('2017 evaluation skipped.')

D:\anaconda3\envs\stormengine\python.exe scripts/evaluate_v7_a.py --config configs/v7_b.yaml --checkpoint artifacts/v7_b_2010_2017/best.pt --scenario clean --device cuda --output-dir artifacts/v7_b_2010_2017/evaluation_2017_clean_seed42
100/547 batches
200/547 batches
300/547 batches
400/547 batches
500/547 batches
{
  "output": "D:\\Documents\\py_projects\\StormEngine-DL\\StormEngine-DL\\artifacts\\v7_b_2010_2017\\evaluation_2017_clean_seed42\\metrics.json",
  "samples": 8743
}
D:\anaconda3\envs\stormengine\python.exe scripts/evaluate_v7_a.py --config configs/v7_b.yaml --checkpoint artifacts/v7_b_2010_2017/best.pt --scenario missing --seed 42 --device cuda --output-dir artifacts/v7_b_2010_2017/evaluation_2017_missing_seed42
100/547 batches
200/547 batches
300/547 batches
400/547 batches
500/547 batches
{
  "output": "D:\\Documents\\py_projects\\StormEngine-DL\\StormEngine-DL\\artifacts\\v7_b_2010_2017\\evaluation_2017_missing_seed42\\metrics.json",
  "samples": 8743
}
D:\anaconda3\env

## 6. Real combined DPC + Open-Meteo replay
Uses only the past 12 valid times from each source. Open-Meteo is model-derived support, not an observation.

In [5]:
if RUN_COMBINED_REPLAY:
    args=['scripts/replay_v7_b.py','--config',CONFIG,'--device',DEVICE]
    if SAVE_PREDICTIONS: args.append('--save-predictions')
    run(*args)
else: print('Combined replay skipped.')

D:\anaconda3\envs\stormengine\python.exe scripts/replay_v7_b.py --config configs/v7_b.yaml --device cuda
windows 25/152
windows 50/152
windows 75/152
windows 100/152
windows 125/152
windows 150/152
windows 152/152
{
  "output": "D:\\Documents\\py_projects\\StormEngine-DL\\StormEngine-DL\\artifacts\\v7_b_2010_2017\\combined_replay_20260801_20260808\\replay_summary.json",
  "windows": 152,
  "all_finite": true
}
